# CS 3892 / 5892 — Session 7 · SAT Solving and Proof by Refutation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/cs3892-2026-09-17-sat-solving-and-refutation.ipynb)

**Thursday, September 17, 2026.** What the solver is doing underneath, in three
examples that run here with nothing installed.

**You never ask a solver whether something is true.** You ask whether it can be
false, and `unsat` is the proof.

## Setup

Run this once.

In [ ]:
# --- Setup: find the repo (clone on Colab), install Z3, define helpers -------
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
SESSION  = "cs3892-2026-09-17-sat-solving-and-refutation"

def _find_repo():
    """Walk up from the CWD looking for the repo; otherwise clone it."""
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "sessions" / SESSION).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "sessions" / SESSION).is_dir():
        print(f"$ git clone {REPO_URL} {dest}")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
os.chdir(ROOT)
print("repo:", ROOT)

try:
    import z3
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver"], check=True)
    import z3
print("Z3", z3.get_version_string())

SM = ROOT / "sessions" / SESSION / "smt2"
PYD = ROOT / "sessions" / SESSION / "python"

def show(path):
    """Print a source file, so you can read what you are about to run."""
    print(f"--- {pathlib.Path(path).name} " + "-" * max(0, 60 - len(pathlib.Path(path).name)))
    print(pathlib.Path(path).read_text().rstrip())
    print()

def run(path, show_source=True):
    """Run one example and stream its output. Raises if it does not pass.

    .smt2 goes through scripts/run_smt2.py, which checks the file's own
    `; EXPECT:` contract. .py is executed directly and asserts internally.
    The pip wheel for Z3 ships no `z3` CLI, which is why .smt2 is run through
    the Python bindings rather than a shell command -- identical everywhere.
    """
    path = pathlib.Path(path)
    if show_source:
        show(path)
    cmd = ([sys.executable, "scripts/run_smt2.py", str(path)] if path.suffix == ".smt2"
           else [sys.executable, str(path)])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.rstrip())
    if r.stderr.strip():
        print(r.stderr.rstrip(), file=sys.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{path} failed")
    return r.stdout

print("ready — helpers: show(path), run(path)")

## 1. Proof by refutation

Slide 24. The contrapositive law proved by hand on the board on September 10:
`(¬q → ¬p) ≡ (p → q)`.

Notice what is actually asserted: not the law, but its **negation** — that the
two sides differ. `unsat` says no interpretation makes them differ, which is
what "valid" means. This is the move every tool in Units 3–10 makes.

In [ ]:
run(SM  / "01_refutation.smt2")
run(PYD / "01_refutation.py")

## 2. `unsat` is not the end of the answer

Slide 30. A checker that says *your policy is inconsistent* is annoying. One
that says *these four rules are the inconsistency* is usable.

Five rules go in; four of them cannot hold together and the fifth is innocent.
The **UNSAT core** names the four. This is what a policy checker has to produce
to be worth running — and it is the same machinery behind the Bedrock-style
automated reasoning checks from session 4.

In [ ]:
run(PYD / "02_unsat_core.py")

## 3. Where the boolean problem comes from

Slide 28. Session 5 asked whether `x + 1 > x` can fail for an 8-bit signed
value, and Z3 said `sat`, `x = 127`. It did not reason about numbers to get
there — it **bit-blasted**: eight booleans for the bits, the adder written out
as clauses, the comparison written out as clauses, then SAT.

One line about an 8-bit number becomes 90 clauses.

In [ ]:
run(PYD / "03_bitblast.py")

## 4. Now break them

1. In `01`, try a law that is **not** valid — `(p → q) ≡ (q → p)`. What does the model tell you?
2. In `02`, delete `alex_is_a_contractor`. Does the core shrink, or does the answer flip?
3. In `02`, add a sixth rule that is also contradictory. Is the core Z3 returns the smallest one? (It is not guaranteed to be.)
4. In `03`, widen the bit-vector to 16 bits and watch the clause count. Then 32.
5. Ask for something genuinely hard: a 20-variable pigeonhole formula. Time it.

In [ ]:
from z3 import *

# Scratch.
p, q = Bool("p"), Bool("q")
s = Solver()
s.add(Not(Implies(p, q) == Implies(q, p)))   # is THIS one valid?
print(s.check())
if s.check() == sat:
    print(s.model())

## Where this goes next

| When | What |
|---|---|
| **Tue Sep 22** | **Quiz 2** (sets, propositional and first-order logic), then SMT proper — DPLL(T), theory solvers, bounded reachability |
| **Thu Sep 24** | **HW1 due** — Z3 from Python |
| **Tue Sep 29** | **Project proposal due** |

Source: [`sessions/cs3892-2026-09-17-sat-solving-and-refutation/`](https://github.com/ttj/cs3892-examples/tree/main/sessions/cs3892-2026-09-17-sat-solving-and-refutation)